In [1]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime
from sqlalchemy import create_engine, text

In [2]:
db_config = {
    'user': 'postgres',
    'pass': '++++', 
    'host': 'localhost',
    'port': '5432',
    'name': 'fuzzy_factory'
}

db_url = f"postgresql://{db_config['user']}:{db_config['pass']}@{db_config['host']}:{db_config['port']}/{db_config['name']}"
engine = create_engine(db_url)
print(f"✅ Connected to: {db_config['name']}")

✅ Connected to: fuzzy_factory


In [3]:
for folder in ['data/bronze', 'data/silver', 'data/gold', 'logs']:
    os.makedirs(folder, exist_ok=True)

logging.basicConfig(filename='logs/etl_process.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("ETL Pipeline Started")

In [4]:
def extract_raw(file):
    df = pd.read_csv(f"data/bronze/{file}")
    logging.info(f"Bronze: Extracted {file} ({len(df)} rows)")
    return df

In [5]:
import zipfile

zip_path = r"C:\Users\Aakarsh Shukla\Downloads\Maven+Fuzzy+Factory.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data/bronze")
    print("✅ Extraction successful!")

✅ Extraction successful!


In [6]:
orders_raw = extract_raw('orders.csv')
products_raw = extract_raw('products.csv')

In [7]:
sessions_raw = extract_raw('website_sessions.csv')
pageviews_raw = extract_raw('website_pageviews.csv')

In [8]:
items_raw = extract_raw('order_items.csv')
refunds_raw = extract_raw('order_item_refunds.csv')

In [9]:
for df in [orders_raw, sessions_raw, pageviews_raw, items_raw, refunds_raw, products_raw]:
    df['created_at'] = pd.to_datetime(df['created_at'])

In [10]:
orders_raw = orders_raw.drop_duplicates(subset=['order_id'])
sessions_raw = sessions_raw.drop_duplicates(subset=['website_session_id'])

In [11]:
orders_raw.to_csv('data/silver/orders_cleaned.csv', index=False)
print("🥈 Silver Layer: Cleaning Complete.")

🥈 Silver Layer: Cleaning Complete.


In [12]:
fact_sales = pd.merge(
    orders_raw,
    sessions_raw[['website_session_id', 'utm_source', 'utm_campaign', 'device_type']],
    on='website_session_id',
    how='left'
)

In [13]:
fact_sales['gross_profit'] = fact_sales['price_usd'] - fact_sales['cogs_usd']
fact_sales['margin_pct'] = (fact_sales['gross_profit'] / fact_sales['price_usd']) * 100

In [14]:
refund_sums = refunds_raw.groupby('order_id')['refund_amount_usd'].sum().reset_index()
fact_sales = pd.merge(fact_sales, refund_sums, on='order_id', how='left').fillna(0)
fact_sales['net_revenue'] = fact_sales['price_usd'] - fact_sales['refund_amount_usd']

In [15]:
gold_marketing_perf = fact_sales.groupby(['utm_source', 'utm_campaign']).agg(
    total_revenue=('net_revenue', 'sum'),
    order_count=('order_id', 'count'),
    avg_margin=('margin_pct', 'mean')
).reset_index()

In [16]:
gold_monthly = fact_sales.set_index('created_at').resample('ME')['net_revenue'].sum().reset_index()

In [17]:
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("🚀 Connection Verified.")

🚀 Connection Verified.


In [18]:
fact_sales.to_sql('fact_sales_performance', engine, if_exists='replace', index=False)

313

In [19]:
gold_marketing_perf.to_sql('agg_channel_roi', engine, if_exists='replace', index=False)
gold_monthly.to_sql('agg_monthly_revenue', engine, if_exists='replace', index=False)

37

In [20]:
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX idx_order_id ON fact_sales_performance(order_id);"))
    conn.commit()
print("⚡ SQL Indexes Created.") 

⚡ SQL Indexes Created.


In [21]:
final_count = pd.read_sql("SELECT COUNT(*) FROM fact_sales_performance", engine).iloc[0,0]
logging.info(f"Pipeline Finished: {final_count} rows loaded.")
print(f"✅ ETL Finished. {final_count} records are now in PostgreSQL.")

✅ ETL Finished. 32313 records are now in PostgreSQL.
